# 04 — Chạy PBE (PySCF PBC) trên tập đã chọn
Lưu mật độ electron, năng lượng PBE và thông tin hội tụ cho từng vật liệu.

- Có **resume**: vật liệu đã có file `.json` được bỏ qua, nên có thể chạy lại sau khi Colab ngắt phiên.
- Chạy từ ô nhỏ đến ô lớn và dừng trước `MAX_WALL_H` giờ để không mất cấu trúc đang chạy dở.

**Đầu ra:** `data/densities/<material_id>.npz` (ρ, ma trận mật độ, cấu trúc, k-point), `<material_id>.json` (năng lượng, hội tụ, thiết lập), `run_log.csv`.

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf pymatgen

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd, time, traceback
from utils import dft_utils as dft, density_utils as du, structure_utils as su

DENS = f"{DATA}/densities"
FORCE_RERUN = False
ONLY_SPLITS = None          # ví dụ ["train"]; None = tất cả
MAX_N = None                # giới hạn số vật liệu mỗi lần chạy (để thử); None = không giới hạn
MAX_WALL_H = 10.5           # dừng nhận việc mới sau số giờ này

cfg = json.load(open(f"{PROC}/pbe_settings.json"))
settings = cfg["settings"]
sel = pd.read_csv(f"{PROC}/selected.csv")
if ONLY_SPLITS:
    sel = sel[sel.split.isin(ONLY_SPLITS)]
sel = sel.sort_values(["n_atoms", "n_elements", "material_id"])     # rẻ trước, đắt sau
print("Thiết lập:", {k: settings[k] for k in ["basis", "pseudo", "xc", "kspacing", "precision", "spin", "conv_tol"]})
print("Số vật liệu:", len(sel))

In [ ]:
t_start = time.time()
log_path = f"{DENS}/run_log.csv"
n_done = n_skip = n_fail = 0

for r in sel.itertuples():
    js, npz = f"{DENS}/{r.material_id}.json", f"{DENS}/{r.material_id}.npz"
    if os.path.exists(js) and os.path.exists(npz) and not FORCE_RERUN:
        n_skip += 1
        continue
    if (time.time() - t_start) / 3600 > MAX_WALL_H or (MAX_N and n_done + n_fail >= MAX_N):
        print("Dừng theo giới hạn thời gian/số lượng; chạy lại notebook để tiếp tục.")
        break
    struct = su.read_structure(f"{RAW}/cif/{r.material_id}.cif")
    try:
        info, rho, dm, kpts, cell = dft.run_scf(struct, r.band_gap_eV, settings)
        du.save_density(npz, rho, struct.lattice.matrix, [s.specie.symbol for s in struct],
                        struct.frac_coords, kpts, dm, info["e_tot_Ha"])
        info.update(material_id=r.material_id, formula=r.formula_pretty, n_atoms=len(struct),
                    band_gap_MP_eV=float(r.band_gap_eV), split=r.split, basis=settings["basis"],
                    pseudo=settings["pseudo"], timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
        du.dump_json(js, info)
        n_done += 1
        print(f"{r.material_id:12s} {r.formula_pretty:10s} conv={info['converged']} e_ok={info['electron_ok']} "
              f"E={info['e_tot_Ha']:.6f} Ha  nk={info['nk']}  {info['wall_s']}s")
        row = {k: info[k] for k in ["material_id", "formula", "converged", "electron_ok", "cycles", "e_tot_Ha", "wall_s", "nk"]}
        pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
        del rho, dm
    except Exception as e:
        n_fail += 1
        print(f"{r.material_id:12s} {r.formula_pretty:10s} LỖI: {type(e).__name__}: {str(e)[:150]}")
        with open(f"{DENS}/failures.log", "a") as f:
            f.write(f"{r.material_id}\t{type(e).__name__}\t{str(e)[:300]}\n")

print(f"\nXong lượt này: chạy mới {n_done}, bỏ qua (đã có) {n_skip}, lỗi {n_fail}")

## Tổng hợp trạng thái

In [ ]:
recs = []
for r in sel.itertuples():
    p = f"{DENS}/{r.material_id}.json"
    if os.path.exists(p):
        i = json.load(open(p))
        recs.append({"material_id": r.material_id, "split": r.split, "converged": i["converged"],
                     "electron_ok": i["electron_ok"], "attempts": i["attempts"], "cycles": i["cycles"],
                     "e_tot_per_atom_Ha": i["e_tot_per_atom_Ha"], "wall_s": i["wall_s"]})
st = pd.DataFrame(recs)
print(f"Đã có kết quả: {len(st)}/{len(sel)}")
if len(st):
    print("Hội tụ:", int(st.converged.sum()), "| electron_ok:", int(st.electron_ok.sum()),
          "| tổng thời gian:", round(st.wall_s.sum() / 3600, 2), "giờ")
    bad = st[~(st.converged & st.electron_ok)]
    print(bad.to_string(index=False) if len(bad) else "Mọi mẫu đều đạt")